# Crowdcal Phase 4: fine-tune `laya-ft` (Laya + RLCD on MNLI subsets)

Trains the `laya-ft` arm of [Crowdcal](https://github.com/AmanSinghNp/crowdcal) (PREREG §4-§5) and selects each run's checkpoint on **dev only**.

**Kaggle settings:** Accelerator **GPU T4 x2**, Internet **On**. The notebook needs no secrets and calls no hosted model API.

**How to run**
1. Open the notebook on Kaggle, set the two options above, and use *Run All*. The crowdcal repo is cloned at the pinned commit `38cd6d8`, and `laya==0.3.21` is installed.
2. All 8 runs are listed in `RUNS` (config cell). To split them over Kaggle sessions, set `ONLY`, for example `ONLY = [(100, 0), (100, 1), (100, 2), (1000, 0), (1000, 1), (1000, 2)]` in one session and `ONLY = [(10000, 0), (30000, 0)]` in another. A run whose `train_meta.json` already exists is skipped.
3. Local smoke test: `DRY_RUN=1` (32 training items, 1 epoch, 50 dev items, one process, device cuda/mps/cpu).
4. Each run trains 4 epochs, saves a checkpoint after every epoch, scores every epoch on `dev` (4,108 items x 3 wordings, raw soft Brier), keeps the best epoch, and deletes the rest. Outputs land in `/kaggle/working/checkpoints/laya-ft@<size>-s<seed>/`, plus `summary.json`.

**What follows the pinned recipe (PREREG §5) and what does not**
- Training loop, RLCD reward, optimizer, learning rates, batch size (8 x 4 accumulation x 2 GPUs = 64), 4 epochs, cosine schedule, exploration noise and the temperature fit come unchanged from the upstream notebook (`NandhaKishorM/laya` @ `9d95567`). Only the data source, the seeds, per-epoch checkpointing and the dev-based selection are new.
- Upstream holds out `min(400, N // 10)` training items for its temperature fit, so a run at size N trains on N minus that (90, 900, 9,600 and 29,600 items). This is kept as is.
- The split guard: only `train@*` and `dev` are ever loaded here. The held-out `calib` and `test` splits are never touched.

## 1. Setup

In [ ]:
# SPLIT GUARD: this notebook may load only train@* and dev. The held-out `calib` and `test` splits are never
# loaded or referenced here; every load goes through load() below, which refuses anything else.
import datetime as dt, json, os, random, shutil, subprocess, sys, time
from pathlib import Path

CROWDCAL_COMMIT = "38cd6d871c5063bb2cf4f50670b2e66d6e4c76ea"
REPO = Path(os.environ.get("REPO_DIR", "/tmp/crowdcal"))
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)

sh('pip install -q "laya==0.3.21" "transformers>=4.48.0" safetensors huggingface_hub httpx')
if not REPO.exists():
    sh(f"git clone -q https://github.com/AmanSinghNp/crowdcal {REPO}")
sh(f"git -C {REPO} checkout -q {CROWDCAL_COMMIT}")
os.chdir(REPO)  # crowdcal reads data/splits/*.ids.txt and config/wordings.json relative to the repo root
sh("pip install -q -e . --no-deps")
assert subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip() == CROWDCAL_COMMIT

import numpy as np, torch, transformers, laya
from crowdcal.data import load_split, load_wordings

ALLOWED_SPLITS = {"train@100", "train@1k", "train@10k", "train@30k", "dev"}
assert not {"calib", "test"} & ALLOWED_SPLITS

def load(name):
    assert name in ALLOWED_SPLITS, f"refusing to load split {name!r}"
    return load_split(name)

print("laya", laya.__version__, "| torch", torch.__version__, "| transformers", transformers.__version__,
      "| GPUs", torch.cuda.device_count())
subprocess.run("nvidia-smi -L", shell=True)

## 2. Config

In [ ]:
DRY_RUN = bool(int(os.environ.get("DRY_RUN", "0")))  # 1: local smoke test (32 train items, 1 epoch, 50 dev items, 1 process)

# (training size, seed). Use ONLY to run a subset in this Kaggle session, e.g. ONLY = [(100, 0), (100, 1), (100, 2)]
RUNS = [(100, 0), (100, 1), (100, 2), (1000, 0), (1000, 1), (1000, 2), (10000, 0), (30000, 0)]
ONLY = None
SPLIT = {100: "train@100", 1000: "train@1k", 10000: "train@10k", 30000: "train@30k"}

BASE_ID, BASE_REV = "convaiinnovations/laya", "55cf4c4ebb4ebe31b2550e8bdf3bd21b99753851"
UPSTREAM_COMMIT = "9d955671415fc19f069b9cc998928075c1f255ec"  # NandhaKishorM/laya, PREREG §4
EPOCHS = 1 if DRY_RUN else 4  # upstream default: 4
TRAIN_CAP, DEV_CAP = (32, 50) if DRY_RUN else (None, None)

OUT = Path(os.environ.get("CROWDCAL_OUT", "/kaggle/working")) / "checkpoints"  # only selected checkpoints land here
WORK = Path(os.environ.get("CROWDCAL_WORK", "/tmp/laya_ft_work"))               # epoch checkpoints, deleted after selection
OUT.mkdir(parents=True, exist_ok=True)
Path('/tmp/laya_ft').mkdir(exist_ok=True)  # the training script is written here by the next cell
WORK.mkdir(parents=True, exist_ok=True)

n_gpu = torch.cuda.device_count()
assert DRY_RUN or n_gpu >= 2, "Set the Kaggle accelerator to GPU T4 x2 (DDP with 2 ranks is part of the recipe)"
if DRY_RUN:
    RUNS = [(100, 0)]
if ONLY:
    RUNS = [r for r in RUNS if r in ONLY]
print("runs this session:", RUNS)

## 3. Training script
The upstream `train_ddp.py` RLCD loop (unchanged hyperparameters), plus per-epoch checkpoints and the dev evaluation. Runs under `torchrun` for DDP.

In [ ]:
%%writefile /tmp/laya_ft/laya_ft.py
"""Laya RLCD fine-tuning for Crowdcal.

The training loop is the upstream `train_ddp.py` (NandhaKishorM/laya @ 9d95567, Kaggle notebook
laya_finetune_typed_decisions_2xT4_kaggle.ipynb) with its hyperparameters unchanged. Only these differ:
  * data comes from Crowdcal (prepared by the notebook), not typed-decisions;
  * seeds: python/numpy/torch, the per-epoch shuffle and the temperature-holdout draw all depend on --seed
    (seed 0 reproduces upstream's constants 42 and 20260922 exactly);
  * a full loadable checkpoint is written after EVERY epoch (upstream keeps one rolling file), each with the
    temperature fitted on the held-out slice at that epoch;
  * it runs without torchrun on one CPU/MPS/CUDA device (dry run); DDP is used when launched by torchrun.

Usage:
  torchrun --standalone --nproc_per_node=2 laya_ft.py train MODEL_DIR RUN_DIR SEED EPOCHS
  python laya_ft.py evaldev OUT_JSON DEV_CAP CKPT_DIR [CKPT_DIR ...]      # dev split only
"""
import json
import os
import random
import sys
import time
from contextlib import nullcontext


def collate_train_batch(items, pad_id):
    import torch
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
        att[i, : len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {
        "input_ids": ids,
        "attention_mask": att,
        "marker_pos": mpos,
        "marker_mask": mmask,
        "target": target,
        "qtype": torch.tensor([it["qtype"] for it in items]),
        "label": torch.tensor([it["label"] for it in items])
    }


def fit_one_temp(sel):
    import torch
    if len(sel) < 10:
        return 1.0
    kmax = max(len(z) for z, _ in sel)
    Z = torch.full((len(sel), kmax), -1e4)
    T = torch.zeros((len(sel), kmax))
    for i, (z, t) in enumerate(sel):
        Z[i, :len(z)] = torch.tensor(z)
        T[i, :len(t)] = torch.tensor(t, dtype=torch.float32)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = -(T * torch.log_softmax(Z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.clamp(log_t.exp(), 0.1, 10.0).item())


def train(model_dir, run_dir, seed, epochs):
    import numpy as np
    import torch
    import torch.distributed as dist
    from torch.nn.parallel import DistributedDataParallel as DDP
    from safetensors.torch import load_file, save_file
    from transformers import AutoTokenizer
    from laya.common import build_model, proper_reward

    use_ddp = "RANK" in os.environ  # set by torchrun
    if use_ddp:
        dist.init_process_group("nccl")
        rank, world_size = dist.get_rank(), dist.get_world_size()
        local_rank = int(os.environ.get("LOCAL_RANK", "0"))
        torch.cuda.set_device(local_rank)
        device = torch.device("cuda", local_rank)
    else:  # dry run: single process on whatever is available
        rank, world_size, local_rank = 0, 1, 0
        device = torch.device("cuda" if torch.cuda.is_available()
                              else "mps" if torch.backends.mps.is_available() else "cpu")
    is_cuda = device.type == "cuda"
    barrier = dist.barrier if use_ddp else (lambda: None)
    amp = (lambda: torch.autocast("cuda", dtype=torch.float16)) if is_cuda else nullcontext

    # crowdcal: seed everything (rank offset so DDP ranks draw different exploration noise)
    random.seed(1000 * seed + rank)
    np.random.seed(1000 * seed + rank)
    torch.manual_seed(1000 * seed + rank)
    torch.cuda.manual_seed_all(1000 * seed + rank)

    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 4096
    cfg["max_len"] = 1024
    cfg["head_max_len"] = 256

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))

    weights = load_file(os.path.join(model_dir, "model.safetensors"))
    model.load_state_dict(weights, strict=True)

    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device)
    model.train()

    ddp_model = DDP(model, device_ids=[local_rank], find_unused_parameters=True) if use_ddp else model

    all_items = torch.load(os.path.join(run_dir, "train_items.pt"), weights_only=False)

    # Hold the temperature-fit slice out of training before sharding (upstream behaviour, kept).
    # NB: this makes the effective training set N - min(400, N // 10) items.
    # crowdcal: holdout seed = upstream's 20260922 + run seed. (Not the Crowdcal `calib` split.)
    TFIT_MAX = 400
    order = list(range(len(all_items)))
    random.Random(20260922 + seed).shuffle(order)
    n_tfit = min(TFIT_MAX, len(all_items) // 10)
    tfit_items = [all_items[i] for i in sorted(order[:n_tfit])]
    train_items = [all_items[i] for i in sorted(order[n_tfit:])]
    my_items = train_items[rank::world_size]

    EPOCHS = epochs      # upstream: 4
    MICRO_BATCH = 8      # 8 sequences per forward pass per GPU
    GRAD_ACCUM = 4       # Effective batch across 2 GPUs = 64 sequences (8 * 2 * 4)
    GROUP_SIZE = 4       # GRPO baseline samples
    LR_ENCODER = 2.5e-5  # Encoder adaptation rate
    LR_HEAD = 1.0e-4     # Head adaptation rate
    SIGMA_START = 0.4    # Exploration noise
    SIGMA_END = 0.1

    enc_params = [p for n, p in ddp_model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in ddp_model.named_parameters() if "encoder." not in n]

    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": LR_ENCODER},
        {"params": head_params, "lr": LR_HEAD}
    ], weight_decay=0.01)

    total_updates = (len(my_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_updates), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=is_cuda)

    def save_epoch_checkpoint(epoch):
        """Rank 0: temperature fit on the held-out slice (upstream procedure), then a complete laya.load()-able dir."""
        torch.cuda.empty_cache()
        model.eval()
        preds = []
        with torch.no_grad():
            for c_idx in range(0, len(tfit_items), 16):
                c_chunk = tfit_items[c_idx:c_idx + 16]
                cb = collate_train_batch(c_chunk, tok.pad_token_id)
                with amp():
                    l_sub, _ = model(
                        cb["input_ids"].to(device),
                        cb["attention_mask"].to(device),
                        cb["marker_pos"].to(device),
                        cb["marker_mask"].to(device),
                        cb["qtype"].to(device)
                    )
                l_np = l_sub.float().cpu().numpy()
                for r, it in enumerate(c_chunk):
                    k = len(it["markers"])
                    preds.append((it["qtype"], l_np[r, :k], it["target"]))
        model.train()
        fitted_temps = [1.2, 1.2, 1.2]
        try:
            for qt in range(3):
                sel = [(z, t) for q_type, z, t in preds if q_type == qt]
                if sel:
                    fitted_temps[qt] = fit_one_temp(sel)
            print("Fitted calibration temperatures (choice, score, noul):", [round(t, 3) for t in fitted_temps])
        except Exception as e:
            print("Temperature fitting fallback:", e)
        out = os.path.join(run_dir, f"epoch_{epoch}")
        os.makedirs(out, exist_ok=True)
        sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
        save_file(sd, os.path.join(out, "model.safetensors"))
        model.encoder.config.save_pretrained(os.path.join(out, "encoder"))
        tok.save_pretrained(os.path.join(out, "tokenizer"))
        c = dict(cfg)
        c["fine_tuned"] = True
        c["model_name"] = "laya-ft-crowdcal"
        c["temperature"] = fitted_temps
        # This fit is per type; inherited bucket overrides would hide the new values.
        c.pop("temperature_by_options", None)
        with open(os.path.join(out, "rl_agent_config.json"), "w") as f:
            json.dump(c, f, indent=2)
        return fitted_temps

    if rank == 0:
        print(f"Starting DDP training (world_size={world_size}): {len(train_items)} train items "
              f"({len(tfit_items)} held out for temperature fit) | {len(my_items)} per rank | {EPOCHS} epochs")
    t0 = time.time()
    log = []

    for epoch in range(EPOCHS):
        random.seed(1000 * seed + 42 + epoch + rank)  # upstream: 42 + epoch + rank
        random.shuffle(my_items)
        epoch_loss, n_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        accum_step = 0

        progress = epoch / max(1, EPOCHS - 1)
        sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * progress

        for b_idx in range(0, len(my_items), MICRO_BATCH):
            chunk = my_items[b_idx:b_idx + MICRO_BATCH]
            if not chunk:
                continue

            batch = collate_train_batch(chunk, tok.pad_token_id)

            with amp():
                logits, act = ddp_model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["marker_pos"].to(device),
                    batch["marker_mask"].to(device),
                    batch["qtype"].to(device)
                )

            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            k = mask.sum(-1, keepdim=True).float()
            target = batch["target"].to(device)

            # 1. Sample G noisy logit distributions with zero-mean projection
            eps = torch.randn((GROUP_SIZE,) + logits.shape, device=device) * sigma * mask
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mask, -1e4), -1)

            # 2. Evaluate proper scoring reward (w_sph=0.75 for soft target matching)
            with torch.no_grad():
                r = proper_reward(q, target.unsqueeze(0), batch["qtype"].to(device), mask, w_sph=0.75, w_rps=1.0)
                adv = r - r.mean(0, keepdim=True)
                adv = adv / (adv.std() + 1e-6)

            # 3. Policy gradient loss + full 1.0 soft cross-entropy guidance
            logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
            loss_rl = -(adv * logp).mean()
            loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
            loss = (loss_rl + 1.0 * loss_ce) / GRAD_ACCUM + 0.0 * act.sum()

            scaler.scale(loss).backward()
            accum_step += 1

            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            epoch_loss += loss.item() * GRAD_ACCUM
            n_batches += 1

            if rank == 0 and (n_batches % 50) == 0:
                cur_lr = scheduler.get_last_lr()[0]
                print(f"  Epoch {epoch+1}/{EPOCHS} | Step {n_batches} | Loss: {loss.item()*GRAD_ACCUM:.4f} | Reward: {r.mean().item():.3f} | LR: {cur_lr:.2e}")

        avg_loss = epoch_loss / max(1, n_batches)
        if rank == 0:
            print(f"=== Epoch {epoch+1}/{EPOCHS} Completed in {time.time()-t0:.1f}s | Avg Loss: {avg_loss:.4f} ===")
        barrier()

        # crowdcal: complete checkpoint after every epoch (rank 0 saves, the other rank waits at the barrier)
        if rank == 0:
            temps = save_epoch_checkpoint(epoch + 1)
            log.append({"epoch": epoch + 1, "avg_loss": avg_loss, "temperature": temps})
            print(f"  Saved epoch checkpoint {epoch+1}/{EPOCHS}")
        barrier()

    train_seconds = time.time() - t0  # excludes model/data loading and (separately run) dev evaluation
    if rank == 0:
        hp = {
            "epochs": EPOCHS, "micro_batch": MICRO_BATCH, "grad_accum": GRAD_ACCUM,
            "effective_batch": MICRO_BATCH * GRAD_ACCUM * world_size, "world_size": world_size,
            "group_size": GROUP_SIZE, "lr_encoder": LR_ENCODER, "lr_head": LR_HEAD,
            "optimizer": "AdamW(weight_decay=0.01)", "scheduler": "CosineAnnealingLR(T_max=total_updates, eta_min=1e-6)",
            "sigma_start": SIGMA_START, "sigma_end": SIGMA_END, "grad_clip": 1.0, "w_sph": 0.75, "w_rps": 1.0,
            "ce_weight": 1.0, "amp": "fp16 autocast + GradScaler" if is_cuda else "none (dry run)",
            "gradient_checkpointing": True, "train_max_len": 1024, "train_head_max_len": 256,
            "tfit_max": TFIT_MAX, "tfit_frac": 0.1, "tfit_holdout_seed": 20260922 + seed,
            "temperature_fit": "LBFGS(lr=0.1, max_iter=100) on log T, clamp [0.1, 10], min 10 items, per question type",
            "shuffle_seed_base": 1000 * seed + 42, "n_train": len(train_items), "n_tfit": len(tfit_items),
            "total_scheduler_steps": total_updates,
        }
        with open(os.path.join(run_dir, "train_stats.json"), "w") as f:
            json.dump({"hyperparameters": hp, "train_seconds": train_seconds, "world_size": world_size,
                       "epochs": log}, f, indent=2)
    barrier()
    if use_ddp:
        dist.destroy_process_group()


def evaldev(out_json, dev_cap, ckpts):
    """Dev soft Brier of each checkpoint via crowdcal's own LocalArm (same laya path + unrounded fix as inference)."""
    import gc
    import numpy as np
    import torch
    from crowdcal.arms import LocalArm
    from crowdcal.data import load_split, load_wordings

    dev = load_split("dev")  # the only non-train split this project touches during fine-tuning
    dev = dev[:dev_cap] if dev_cap else dev
    words = load_wordings()
    res = {}
    for ck in ckpts:
        arm = LocalArm("laya-ft", ck, "local", True)
        arm.predict(dev[0], words[0])  # load + warm up, untimed
        per_item, secs = [], 0.0
        for it in dev:
            se = []
            for w in words:
                t = time.perf_counter()
                p = arm.predict(it, w).p_yes  # returns after the numpy conversion, i.e. GPU work is done
                secs += time.perf_counter() - t
                se.append((p - it.q) ** 2)
            per_item.append(np.mean(se))
        res[ck] = {"dev_soft_brier": float(np.mean(per_item)), "n_items": len(dev),
                   "infer_sec_per_1k": 1000 * secs / (len(dev) * len(words))}
        print(ck, res[ck], flush=True)
        del arm
        gc.collect()
        torch.cuda.empty_cache()
    with open(out_json, "w") as f:
        json.dump(res, f)


if __name__ == "__main__":
    if sys.argv[1] == "train":
        train(sys.argv[2], sys.argv[3], int(sys.argv[4]), int(sys.argv[5]))
    else:
        evaldev(sys.argv[2], int(sys.argv[3]), sys.argv[4:])

## 4. Train, evaluate on dev, select, save
Per run: build the training set, train 4 epochs, score every epoch on dev, keep the lowest dev soft Brier.

In [ ]:
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from laya.agent import _fix_tokenizer_config
from laya.common import build_sequence, render_options, QTYPES

SCRIPT = Path("/tmp/laya_ft/laya_ft.py")
assert SCRIPT.exists(), "run the previous (%%writefile) cell first"

# English root checkpoint of the pinned base revision (same allow_patterns as laya.load uses)
model_dir = snapshot_download(BASE_ID, revision=BASE_REV,
                              allow_patterns=["rl_agent_config.json", "model.safetensors", "tokenizer/*", "encoder/*"])
_fix_tokenizer_config(model_dir)
tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
cfg = json.load(open(os.path.join(model_dir, "rl_agent_config.json")))
WORDINGS = load_wordings()  # the 3 frozen wordings
CONTEXT_TEMPLATE = json.load(open("config/wordings.json"))["context_template"]  # exactly what crowdcal.arms.LocalArm uses


def build_training_item(state, q, gold_q):
    """Upstream preprocessing (noul branch only): same target / label / build_sequence call."""
    t = q["type"]
    crit = q.get("criteria", {})
    target = [gold_q["probabilities"].get("false", 0.5), gold_q["probabilities"].get("true", 0.5)]
    s = sum(target)
    target = [v / s for v in target] if s > 0 else [1.0 / len(target)] * len(target)
    label = target.index(max(target))
    k = len(render_options({"t": t, "crit": crit}))
    seq, markers = build_sequence(tok, state, {"t": t, "ins": q["instructions"], "crit": crit}, cfg["max_len"], cfg["head_max_len"])
    if len(markers) != k:
        return None
    return {"ids": seq, "markers": markers, "qtype": QTYPES[t], "target": target, "label": label}


def prepare(items, seed, run_dir):
    """One Noul question per item, wording drawn uniformly (seeded) from the 3 frozen wordings; gold = hard label."""
    rng = random.Random(seed)
    out = []
    for it in items:
        state = CONTEXT_TEMPLATE.format(premise=it.premise, hypothesis=it.hypothesis)  # as in LocalArm.predict
        q = {"type": "noul", "instructions": WORDINGS[rng.randrange(len(WORDINGS))].text}
        gold = {"probabilities": {"false": 1.0 - it.label, "true": float(it.label)}}
        out.append(build_training_item(state, q, gold))
    assert all(o is not None for o in out), "an item exceeded the head budget"
    torch.save(out, run_dir / "train_items.pt")


def train(run_dir, seed):
    if DRY_RUN or n_gpu < 2:
        cmd = [sys.executable, str(SCRIPT)]
    else:
        cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", "--nproc_per_node=2", str(SCRIPT)]  # = torchrun
    subprocess.run(cmd + ["train", model_dir, str(run_dir), str(seed), str(EPOCHS)], check=True)


def dev_eval(ckpts, run_dir):
    """Dev soft Brier per epoch checkpoint. With 2 GPUs the checkpoints are split over one process per T4."""
    groups = [ckpts[i::n_gpu] for i in range(n_gpu)] if n_gpu >= 2 and not DRY_RUN else [ckpts]
    procs = []
    for i, g in enumerate(groups):
        env = {**os.environ, **({"CUDA_VISIBLE_DEVICES": str(i)} if len(groups) > 1 else {})}
        procs.append(subprocess.Popen([sys.executable, str(SCRIPT), "evaldev", str(run_dir / f"dev_{i}.json"),
                                       str(DEV_CAP or 0), *map(str, g)], env=env))
    assert all(p.wait() == 0 for p in procs), "dev evaluation failed"
    return {k: v for i in range(len(groups)) for k, v in json.load(open(run_dir / f"dev_{i}.json")).items()}


def run_one(size, seed):
    name = f"laya-ft@{size}-s{seed}"
    final = OUT / name
    if (final / "train_meta.json").exists():
        print("already done:", name)
        return
    print(f"\n===== {name} =====")
    started = dt.datetime.now(dt.timezone.utc).isoformat()
    run_dir = WORK / name
    shutil.rmtree(run_dir, ignore_errors=True)
    run_dir.mkdir(parents=True)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

    items = load(SPLIT[size])[:TRAIN_CAP]
    prepare(items, seed, run_dir)
    train(run_dir, seed)
    stats = json.load(open(run_dir / "train_stats.json"))

    ckpts = [run_dir / f"epoch_{e}" for e in range(1, EPOCHS + 1)]
    dev = dev_eval(ckpts, run_dir)
    epochs = [{"epoch": e, "dev_soft_brier": dev[str(c)]["dev_soft_brier"]} for e, c in enumerate(ckpts, 1)]
    best = min(epochs, key=lambda r: r["dev_soft_brier"])["epoch"]  # dev only; ties -> earliest epoch
    print("dev soft Brier by epoch:", [(r["epoch"], round(r["dev_soft_brier"], 5)) for r in epochs], "-> keep epoch", best)

    sec_per_1k = dev[str(ckpts[best - 1])]["infer_sec_per_1k"]  # measured on the kept checkpoint's dev pass, one T4
    shutil.rmtree(final, ignore_errors=True)
    shutil.move(str(run_dir / f"epoch_{best}"), str(final))  # dir layout == what laya.load(<dir>) reads
    for f in ("model.safetensors", "rl_agent_config.json", "encoder/config.json", "tokenizer/tokenizer_config.json"):
        assert (final / f).exists(), f
    shutil.rmtree(run_dir)  # drops the other epoch checkpoints and train_items.pt

    meta = {
        "arm": "laya-ft", "size": size, "seed": seed,
        "base": f"{BASE_ID}@{BASE_REV}",
        "upstream_notebook_commit": UPSTREAM_COMMIT,
        "crowdcal_commit": CROWDCAL_COMMIT,
        "hyperparameters": stats["hyperparameters"],
        "epochs": epochs, "selected_epoch": best,
        "gpu_hours": stats["train_seconds"] / 3600 * stats["world_size"],  # wall-clock training hours x GPUs, no dev eval
        "infer_sec_per_1k": sec_per_1k,
        "versions": {"torch": torch.__version__, "transformers": transformers.__version__, "laya": laya.__version__,
                     "python": sys.version.split()[0]},
        "dry_run": DRY_RUN,
        "started": started, "finished": dt.datetime.now(dt.timezone.utc).isoformat(),
    }
    (final / "train_meta.json").write_text(json.dumps(meta, indent=2))
    du = shutil.disk_usage(OUT)
    print(f"saved {final} | {sum(f.stat().st_size for f in final.rglob('*') if f.is_file())/1e9:.2f} GB | "
          f"disk used {du.used/1e9:.1f} GB, free {du.free/1e9:.1f} GB")


for size, seed in RUNS:
    run_one(size, seed)

## 5. Summary

In [ ]:
metas = [json.load(open(p)) for p in sorted(OUT.glob("*/train_meta.json"))]
summary = {"n_runs": len(metas), "runs": [
    {k: m[k] for k in ("size", "seed", "selected_epoch", "epochs", "gpu_hours", "infer_sec_per_1k", "finished")}
    | {"dir": f"laya-ft@{m['size']}-s{m['seed']}",
       "selected_dev_soft_brier": next(e["dev_soft_brier"] for e in m["epochs"] if e["epoch"] == m["selected_epoch"])}
    for m in metas]}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2))
for r in summary["runs"]:
    print(f"{r['dir']:>22}  epoch {r['selected_epoch']}  dev soft Brier {r['selected_dev_soft_brier']:.5f}  "
          f"{r['gpu_hours']:.2f} GPU-h  {r['infer_sec_per_1k']:.1f} s/1k")
print("total GPU-hours:", round(sum(r["gpu_hours"] for r in summary["runs"]), 2))
sh(f"du -sh {OUT} && df -h {OUT} | tail -1")

## Download the outputs

Everything you need is in `/kaggle/working/checkpoints/`: one directory per run (`laya-ft@100-s0`, ...), each holding the selected checkpoint and `train_meta.json`, plus `summary.json`.

1. In the Kaggle editor use **Save Version** (Save & Run All, commit). When it finishes, open the version, go to the **Output** tab, and download `checkpoints/` (a zip). Or, from a terminal with the Kaggle CLI set up:
   `kaggle kernels output <your-user>/<this-notebook-slug> -p ./kaggle_out`
2. Put each run directory in the crowdcal repo at `data/checkpoints/<dir>/`, for example `data/checkpoints/laya-ft@100-s0/`. Session outputs from different versions can be merged, since the directory names never collide. Keep `summary.json` beside them.
3. Check that a checkpoint loads: `uv run python -c "import laya; laya.load('data/checkpoints/laya-ft@100-s0')"`.

`data/checkpoints/` is git-ignored. Record `train_meta.json` and `summary.json` in the results write-up (dev soft Brier per epoch, GPU-hours, seconds per 1k predictions).